# 03 – Frage 2: Mieten nach Kanton und Zimmerzahl

**Leitfrage:** Welche Kantone sind bei den Mieten Spitzenreiter, welche Nachzügler — und wo steht Zürich?

- **Teil 1:** Momentaufnahme 2024 (Niveau)
- **Teil 2:** Entwicklung 2012–2024 (Veränderung über die Zeit)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
RAW = "../data/raw"
PROCESSED = "../data/processed"

# Gemeinsame Hilfsfunktion: ein Jahres-Tabellenblatt (Kanton x Zimmerzahl) -> Tidy
WERT_SPALTEN = [1, 3, 5, 7, 9, 11, 13]  # jede 2. Spalte ist ein Vertrauensintervall
ZIMMER = ["Total", "1", "2", "3", "4", "5", "6+"]

def parse_sheet(pfad, sheet, jahr=None):
    raw = pd.read_excel(pfad, sheet_name=sheet, header=None)
    zeilen = []
    for i in range(5, raw.shape[0]):
        name = raw.iloc[i, 0]
        if not isinstance(name, str) or pd.isna(raw.iloc[i, 1]):
            continue  # Fussnoten / Leerzeilen
        rec = {"Kanton": name.strip()}
        if jahr is not None:
            rec["Jahr"] = jahr
        for c, z in zip(WERT_SPALTEN, ZIMMER):
            rec[z] = pd.to_numeric(raw.iloc[i, c], errors="coerce")
        zeilen.append(rec)
    return pd.DataFrame(zeilen)

## Teil 1 – Momentaufnahme 2024

Datensatz `je-d-09.03.03.05.xlsx` = durchschnittlicher Mietpreis **pro m²** nach Kanton und Zimmerzahl (nur 2024).

In [ ]:
df = parse_sheet(f"{RAW}/je-d-09.03.03.05.xlsx", 0)
df.to_csv(f"{PROCESSED}/miete_kanton_zimmer_2024.csv", index=False)

ch = df[df["Kanton"] == "Schweiz"].iloc[0]
kantone = df[df["Kanton"] != "Schweiz"].copy()
print(f"{len(kantone)} Kantone | CH-Schnitt: {ch['Total']} Fr./m²")
df.head()

### Ranking der Kantone (Total)

In [ ]:
d = kantone.sort_values("Total")
farben = ["#c8102e" if k == "Zürich" else "#5a7d99" for k in d["Kanton"]]
plt.figure(figsize=(8, 9))
plt.barh(d["Kanton"], d["Total"], color=farben)
plt.axvline(ch["Total"], color="gray", ls="--", lw=1)
plt.text(ch["Total"] + 0.1, 0.2, f"CH-Schnitt {ch['Total']}", color="gray", fontsize=9)
plt.title("Durchschnittsmiete pro m² nach Kanton (2024)\nZürich hervorgehoben")
plt.xlabel("Franken pro m² / Monat (Nettomiete)")
plt.tight_layout(); plt.savefig("../figures/miete_kanton_ranking_2024.png", dpi=130); plt.show()

### Heatmap Kanton × Zimmerzahl

In [ ]:
hm = kantone.set_index("Kanton")[["1", "2", "3", "4", "5", "6+"]].sort_values("3", ascending=False)
plt.figure(figsize=(8, 10))
sns.heatmap(hm, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "Fr./m²"})
plt.title("Miete pro m² nach Kanton und Zimmerzahl (2024)")
plt.xlabel("Zimmerzahl"); plt.ylabel("")
plt.tight_layout(); plt.savefig("../figures/miete_kanton_zimmer_heatmap_2024.png", dpi=130); plt.show()

**Key Findings Teil 1 (2024):** Teuerste Kantone Zug (21,5), **Zürich (21,3)**, Genf (21,1). Günstigste Jura (12,3), Neuenburg (13,7), Appenzell A.Rh. (13,9). Zürich ist **Rang 2/26** und ~20 % über dem CH-Schnitt. Kleine Wohnungen kosten pro m² am meisten.

## Teil 2 – Entwicklung 2012–2024

Quelle: BFS-Tabelle `miete_m2_kanton_zimmer_2012-2022.xlsx` (Asset 30885400) — **ein Tabellenblatt pro Jahr 2012–2022**. Das aktuellste Jahr 2024 ergänzen wir aus der Snapshot-Datei aus Teil 1 (gleiche Metrik Fr./m²).

> Hinweis: 2023 fehlt in den offenen Tabellen; ab 2018 gab es methodische Anpassungen, die Werte sind aber pro m² gut vergleichbar.

In [ ]:
ts_datei = f"{RAW}/miete_m2_kanton_zimmer_2012-2022.xlsx"
frames = [parse_sheet(ts_datei, str(j), j) for j in range(2012, 2023)]
frames.append(parse_sheet(f"{RAW}/je-d-09.03.03.05.xlsx", 0, 2024))  # 2024 anhängen

zeit = pd.concat(frames, ignore_index=True)
zeit.to_csv(f"{PROCESSED}/miete_kanton_zeitreihe.csv", index=False)
print("Jahre:", sorted(zeit["Jahr"].unique()))
zeit.head()

### Verlauf ausgewählter Kantone

In [ ]:
auswahl = ["Zürich", "Zug", "Genf", "Schweiz", "Bern", "Jura"]
plt.figure(figsize=(9, 5.5))
for k in auswahl:
    d = zeit[zeit["Kanton"] == k].sort_values("Jahr")
    ls = "--" if k == "Schweiz" else "-"
    lw = 2.5 if k in ("Zürich", "Schweiz") else 1.4
    plt.plot(d["Jahr"], d["Total"], marker="o", ms=3, ls=ls, lw=lw, label=k)
plt.title("Durchschnittsmiete pro m² nach Kanton, 2012–2024")
plt.xlabel("Jahr"); plt.ylabel("Franken pro m² / Monat"); plt.legend()
plt.tight_layout(); plt.savefig("../figures/miete_kanton_zeitreihe.png", dpi=130); plt.show()

### Wer ist am stärksten gestiegen? (2012 → 2024)

In [ ]:
piv = zeit.pivot_table(index="Kanton", columns="Jahr", values="Total")
veraenderung = ((piv[2024] / piv[2012] - 1) * 100).round(1)
ch_chg = veraenderung["Schweiz"]
veraenderung = veraenderung.drop("Schweiz").sort_values()

farben = ["#c8102e" if k == "Zürich" else "#5a7d99" for k in veraenderung.index]
plt.figure(figsize=(8, 9))
plt.barh(veraenderung.index, veraenderung.values, color=farben)
plt.axvline(ch_chg, color="gray", ls="--", lw=1)
plt.text(ch_chg + 0.15, 0.2, f"CH +{ch_chg}%", color="gray", fontsize=9)
plt.title("Anstieg der Miete pro m² 2012 → 2024 nach Kanton\nZürich hervorgehoben")
plt.xlabel("Veränderung in %")
plt.tight_layout(); plt.savefig("../figures/miete_kanton_anstieg.png", dpi=130); plt.show()

### Key Findings Teil 2 (2012 → 2024)

- **Die Mieten stiegen schweizweit um rund +12 % pro m².**
- **Am stärksten:** Glarus (+19,5 %), Basel-Stadt (+16,6 %), **Zürich (+15,1 %)**.
- **Am wenigsten:** Appenzell A.Rh. (+6,9 %), Nidwalden (+7,6 %), Graubünden (+7,8 %).
- **Zürich liegt nicht nur im Niveau (Rang 2), sondern auch beim Anstieg vorne (Rang 3/26)** — die teuren Kantone wurden tendenziell noch teurer.
- Der Abstand zwischen teuren (Zürich, Zug, Genf) und günstigen Kantonen (Jura) bleibt über die Jahre stabil gross.